## Introduction

In this tutorial, we will build a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. We will train the model on the text from "warandpeace.txt". This project will help you understand how RNNs can be implemented for text generation tasks and their application in building your own autocomplete model.


## Importing Necessary Libraries

In [2]:
# This is Cell #1

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re


## Setting Up the Device

In [3]:
# This is Cell #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Reading and Preprocessing the Data

Now it is time to prepare our training data.


In [4]:
# This is Cell #3

def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

sequence = read_file("warandpeace.txt")


### Here we will train our model with a simple sequence

We will start by training our model with a simple sequence and repettitive sequence such as `"abcdefghijklmnopqrstuvwxyzabcdef..."`, and we will see if our RNN is capable of learning that pattern or not. This will help you easily verify if your RNN is working correctly or not.

In [58]:
# This is Cell #4

sequence = "abcdefghijklmnopqrstuvwxyz" * 100

## Create Character Mappings

Creating character mappings is essential because RNNs require numerical input to process data. By mapping each unique character to an index and creating a reverse mapping, we convert text data into numerical sequences that the model can understand. This step allows us to encode input text for training and decode the model's output back into readable characters during text generation.



In [69]:
# This is Cell #5

#TODO: Create a list of unique characters from the text sequence
vocab = sorted(set(sequence))

#TODO: Create two dictionaries for character-index mappings that map each character in vocab to a unique index and vice versa
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}

#TODO: Convert the entire text based data into numerical data
data = [char_to_idx[char] for char in sequence]

## Defining the CharDataset Class

Now we will create a custom dataset class to generate sequences and targets for training

Creating a custom `CharDataset` class is crucial because it prepares our text data into input sequences and target sequences that the RNN can learn from. By organizing the data this way, we can efficiently feed batches of sequences into the model during training, allowing it to learn the patterns of character sequences in the text.

In [23]:
# This is Cell #6

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []
        
        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target
    

## Setting Hyperparameters

Now we will set our model's hyperparameters for our training process

Setting hyperparameters is important because they define the model's architecture and training behavior. They determine how the RNN processes data, learns patterns, and how quickly it converges during training. Properly chosen hyperparameters can significantly improve model performance and is a key step in training of models

Set the following hyperparameters for your model in the code cell below:
`sequence_length`, `stride`, `embedding_dim`, `hidden_size`, `num_layers`, `learning_rate`, `num_epochs`, `batch_size`, `vocab_size`.

In [71]:
# This is Cell #7

#TODO: Set your model's hyperparameters

sequence_length = 100  # Length of each input sequence
stride = 10            # Stride for creating sequences
embedding_dim = 64     # Dimension of character embeddings
hidden_size = 128      # Number of features in the hidden state of the RNN
learning_rate = 0.001  # Learning rate for the optimizer
num_epochs = 2         # Number of epochs to train
batch_size = 64        # Batch size for training
vocab_size = len(vocab)
input_size = len(vocab)
output_size = len(vocab)


After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.

TODO: Explain below
1. `sequence_length` determines the number of characters fed into the RNN at one time.
2. `stride` is the step size for moving the input window across the data.
3. `embedding_dim` specifies the size of the vector representation for each character.
4. `hidden_size` is the number of features in the RNN's hidden state.
5. `learning_rate` controls how much to adjust the model weights with respect to the loss gradient.
6. `num_epochs` is the number of complete passes through the training dataset.
7. `batch_size` is the number of training examples utilized in one iteration.
8. `vocab_size` is the total number of unique characters in the sequence.
9. `input_size` corresponds to the vocab size, defining the output layer's dimensionality, allowing the model to predict the next character accurately.
10. `output_size` also corresponds to the vocab size, defining the output layer's dimensionality, allowing the model to predict the next character accurately.

## Splitting Data into Training and Testing Sets

By now at this point in class, I'm confident that you know why we do this, so I'm not gonna say a lot here, let's jump right into the todo.

In [72]:
# This is Cell #8

data_tensor = torch.tensor(data, dtype=torch.long)

#TODO: Convert the data into a pytorch tensor and split the data into 90:10 ratio
train_size = int(0.9 * len(data_tensor))
train_data = data_tensor[:train_size]
test_data = data_tensor[train_size:]


## Creating Data Loaders

Now we will create data loaders for easy batching during training and testing.

Creating data loaders is essential to batch the data during training and testing. Batching allows the RNN to process multiple sequences in parallel, which speeds up training and makes better use of computational resources. 
We will also use Data loaders to shuffle the batched data, which is important for training models that generalize well.

Make sure to set `drop_last=True`

In [73]:
# This is Cell #9

train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)

#TODO: Initialize the training and testing data loader with batching and shuffling equal to True for training (and shuffling = False for testing)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

## Defining the RNN Model

Here we will define our character-level RNN model.

In [41]:
# This is Cell #10

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) 
        #TODO: set the fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation from the lecture 
            # We add a bias as well to expand the range of learnable functions
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v] 
        return logits, final_hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)


For a basic high level understanding of what is the CharRNN model that you just defined above, it consists of an embedding layer, an RNN layer, and a fully connected layer. Then embedding layer converts character indices into embeddings. Then RNN processes the embeddings and captures sequential information. Then finally the fully connected layer maps the RNN outputs to the vocabulary size for character prediction.


# Initializing the Model, Loss Function, and Optimizer

Now we will create an instance of the model that we just defined above and set up the loss function and optimizer. Then we will define a loss function, that evaluates the model's prediction against the true targets, and attaches a cost (number) on how good/bad the model is doing. During our training process, it is this cost that we try to minimize by tweaking the weights of the network. 

Then we will set up an optimizer, which will update the model's parameters based on the loss returned by the our loss function. This is how our model will learn over time.


In [74]:
# This is Cell #12

#TODO: Initialize your RNN model
model = CharRNN(input_size, hidden_size, output_size, embedding_dim)

#TODO: Define the loss function (use cross entropy loss)
criterion = nn.CrossEntropyLoss()

#TODO: Initialize your optimizer passing your model parameters and training hyperparameters
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## Training the Model

Now finally, after all the setup that we have done, we can train our RNN. 

A basic idea high level idea of what we will do here is we will loop over epochs and batches to train the model. 
We will Initialize the hidden state at the beginning of each epoch. For each batch, we will reset the gradients, perform a forward pass, compute the loss, perform backpropagation, and update the model parameters. Then we detach the hidden state to prevent gradients from backpropagating through previous batches. We ill repeat this process for each batch. And finally we will calculate the average loss and accuracy for each epoch.
By performing forward and backward passes, calculating loss, and updating the model parameters, we enable the RNN to improve its predictions with each epoch.

In [75]:
# This is Cell #13

for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/2:   0%|          | 0/3 [00:00<?, ?it/s]C:\Users\spops\AppData\Local\Temp\ipykernel_15652\1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
C:\Users\spops\AppData\Local\Temp\ipykernel_15652\1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/2: 4387it [05:38, 12.98it/s]                    


Epoch [1/2], Loss: 1.7235, Accuracy: 49.54%


Epoch 2/2: 4387it [06:48, 10.75it/s]                    

Epoch [2/2], Loss: 1.5635, Accuracy: 54.12%


## Check your loss

The training loss of your model when trained with a simple sequence like `"abcdefghijklmnopqrstuvwxyz" * 100` should be extremely close to zero. If that's not the case, go back and fix your bugs ;)

If you have acheived a training loss of 0 or extremley close to 0, then congratulations, lets move on to train your model with a bit more complicated sequence. That is our old favorite book, `warandpeace.txt`.

### Read the `warandpeace.txt` file

In [68]:
# This is Cell #14

sequence = read_file('warandpeace.txt')

### Now Follow the instructions

1. Re-run Cell #5 to re-create character mappings for `warandpeace.txt`
2. Re-run Cell #7 to re-initialize hyperparameters
3. Re-run Cell #8 to split and create training and testing data with `warandpeace.txt` as your corpus
4. Re-run Cell #9 to set up data loaders with `warandpeace.txt` data
5. Re-run Cell #12 to re-initialize a new model object (maybe ask yourself why can't you use the previous model that was trained on the simple `"abc..."` corpus)
6. Re-run Cell #13 to train the new model with `warandpeace.txt` data.
   

## Evaluating the Model

After training, we evaluate the model on the test data.

In [76]:
# This is Cell #15

with torch.no_grad():
    #TODO: Write the testing loop for your trained model by refering to the training loop code given to you above
    total_test_loss, correct_predictions, total_predictions = 0, 0, 0
    hidden = model.init_hidden(batch_size)  # Initialize hidden state

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Testing"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets

        # Calculate accuracy
        _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

        correct_predictions += (predicted_indices == batch_targets).sum().item()
        total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_test_loss += loss.item()

    avg_test_loss = total_test_loss / len(test_loader)
    test_accuracy = correct_predictions / total_predictions * 100  # Convert to percentage

    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

Testing:   0%|          | 0/487 [00:00<?, ?it/s]C:\Users\spops\AppData\Local\Temp\ipykernel_15652\1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
C:\Users\spops\AppData\Local\Temp\ipykernel_15652\1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Testing: 100%|██████████| 487/487 [00:08<00:00, 59.52it/s]

Test Loss: 1.5635, Test Accuracy: 54.12%


## Generating Text with the Trained Model

In this part of the assignment, your task is to implement the `generate_text` function, which uses a trained RNN model to generate text character-by-character, continuing from a given input. The function will produce an extended sequence by repeatedly predicting and appending the next character to the input.

### What the function is supposed to do?

1. Take an initial input text of length `n` from the user, convert it into indices using a predefined vocabulary (char_to_idx).
2. Use a trained model to predict the next character in the sequence.
3. Append the predicted character to the input, extend the input sequence, and repeat the process until `k` additional characters are generated.
4. Return the generated text, including the original input and the newly predicted characters.


In [78]:
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)
    
    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
        model: The trained RNN model used for character prediction.
        start_text: The initial string of length `n` provided by the user to start the generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: Optional
        A scaling factor for randomness in predictions. Higher values (e.g., >1) make 
            predictions more random, while lower values (e.g., <1) make predictions more deterministic.
            Default is 1.0.
    """
    start_text = start_text.lower()
    #TODO: Implement the rest of the generate_text function
    input_indexes = [char_to_idx[char] for char in start_text]
    input_tensor = torch.tensor(input_indexes, dtype=torch.long).unsqueeze(0).to(device)
    
    generated_text = start_text

    hidden = model.init_hidden(1)  # Initialize hidden state

    for _ in range(k):
        output, hidden = model(input_tensor, hidden)
        last_logits = output[:, -1, :]
        next_char_idx = sample_from_output(last_logits, temperature).item()
        input_indexes.append(next_char_idx)
        input_tensor = torch.tensor(input_indexes[-n:]).unsqueeze(0).to(device)
        generated_text += idx_to_char[next_char_idx]


    return generated_text

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':
        print("Exiting...")
        break
    
    n = len(start_text) 
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0
    
    completed_text = generate_text(model, start_text, n, k, temperature)
    
    print(f"Generated text: {completed_text}")

Training complete. Now you can generate text.
Generated text: steps and had been and the first that he was a stood and the countess and was the same of the count the french and he was and was the words of the countess and the speaking and the sound the countess and and the french and a started his said the same of the dispent before the dear and he was all the same the countess and some of the count and the countess have seemed to the sound the emperor and was seemed the countess and such a commander and had already and the passing the bright of the regiment that the countess and a state him the same the countess and the countess and continued to the countess and the french and his said the countess and the distance and have the passed his served the same ready and all the princess mary and the same respection of the some of the same and was and some of the same proving and she had been at the count and the count the countess and the countess with his said the emperor the table and the

## Report section

In your report, describe your experiments and observations when training the model with two datasets: (1) the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` and (2) the text from `warandpeace.txt`. Include the final loss values for both datasets and discuss how the generated text differed between the two. Explain the impact of changing the `temperature` parameter on the text generation, and provide examples. Reflect on the challenges you faced, your thought process during implementation, and the key insights you gained about RNNs and sequence modeling.


### ABC Sequence

**Final Loss Values**:
1. **Epoch 1**: Loss: 1.4216, Accuracy: 65.17%
2. **Epoch 2**: Loss: 0.0035, Accuracy: 99.93%

**Generated Text**:
1. **Example 1 Starting Text**: 'a'
    - **Text**: abcdefghijklmnopqrstuvwxyza
    - **Temperature**: 0.1
2. **Example 2 Starting Text**: 'a'
    - **Text**: abcdefghijklmnopqvstuvwxyza
    - **Temperature**: 0.5
3. **Example 3 Starting Text**: 'a'
    - **Text**: abijqmjdqijymmnpqcijklmvoqz
    - **Temperature**: 1.0
4. **Example 4 Starting Text**: 'a'
    - **Text**: ausppphhstwqixsnpqeuxyuaoun
    - **Temperature**: 1.5
5. **Example 5 Starting Text**: 'that'
    - **Text**: thatcdefghijklmnopqrstuvwxyzab
    - **Temperature**: 0.1
6. **Example 6 Starting Text**: 'that'
    - **Text**: thatuvwxyzabcdefghijklmnopqrst
    - **Temperature**: 0.25
7. **Example 7 Starting Text**: 'that'
    - **Text**: thatklmnopqrstuvwxyzabcdefghij
    - **Temperature**: 0.5

### War and Peace Sequence

**Final Loss Values**:
1. **Epoch 1**: **Loss**: 1.7206, **Accuracy**: 49.67%
2. **Epoch 2**: **Loss**: 1.5610, **Accuracy**: 54.17%

**Generated Text**:
1. **Example 1 Starting Text**: 'a'
    - **Text**: and he had been and the same the same the countess and the same the commander and the same the same t
    - **Temperature**: 0.1
2. **Example 2 Starting Text**: 'a'
    - **Text**: and the french in it was as he is it to a father to the ready to do give the fire of the princess mar
    - **Temperature**: 0.5
3. **Example 3 Starting Text**: 'a'
    - **Text**: and the tarce sayboustened to his suitisionly condin, sarm. lowllopens in past by it,he several bee m
    - **Temperature**: 1.0
4. **Example 4 Starting Text**: 'a'
    - **Text**: ag to vilfest him mr! ridging, taquing!dont. ecmintths scornwas time besomething toill helocitietject
    - **Temperature**: 1.5
5. **Example 5 Starting Text:** 'that'
    - **Text**: that a man army and the rostv and the said the was a stook the had begand the was a man army the was a s
    - **Temperature**: 0.1
6. **Example 6 Starting Text**: 'that'
    - **Text**: that a left of his ever to and scourd with all and plack of them to dones angry spossing cappress of the
    - **Temperature**: 0.5
7. **Example 7 Starting Text**: 'that'
    - **Text**: that going this, said a passion he had rone, remaining alehononsone reple.in know kutzov, hous; whecrita
    - **Temperature**: 1.0
8. **Example 8 Starting Text**: 'that'
    - **Text**: that or he uslich.itouched aches hudinsfov is natives cleans.belonallyah,antesoctarned anxie men dom:com
    - **Temperature**: 1.5
9. **Example 9 Starting Text**: 't'
    - **Text**: the street to his head and the same the same the same the same the same the same the same the same th
    - **Temperature**: 0.001
10. **Example 10 Starting Text**: 't'
    - **Text**: the rest such the countess and with his shoulders of his standing the same the same the truel and had
    - **Temperature**: 0.25
11. **Example 11 Starting Text**: 't'
    - **Text**: the french and the countess and the same the countess of the street to the soldiers was a standing to
    - **Temperature**: 0.15
12. **Example 12 Starting Text**: 't'
    - **Text**: the hand, and the princess mary hands possible to the talk of the same to his head and the countess t
    - **Temperature**: 0.45

### Discussion
**ABCs Conclusion**
- The ABC sequence trained very well with a final loss of only 0.0035 and a very good accuracy of 99.93%.
- While generating text, I decided to try both a single letter and a whole word to test what the outputs would be for 26 predicted characters.
- During the single letter tests, with a temperature of 0.1 and 0.5, the generated text perfectly followed the alphabet. However, once I got past 0.5 for temperature, with every increase in the temperature, the following predicted characters got more and more random and farther from the original sequence. 
- While performing the test with a full word, when the temperature was at 0.5, it appeared the model chose a random letter to follow the word "that" but then got back into simply giving the next character in the ABCs. This is different from when the temperature was set to 0.1 and 0.25 because when it was 0.1, it appears to have seen the 'a' in "that" and then pretended the second 't' in that was a 'b' and continued the sequence with a 'c'. As for the generated outcome for the 0.25 temperature, it saw the final 't' in 'that' and continued from 'u' which is the following character in the alphabet.

**War and Peace Conclusion**
- The War and Peace sequence did not train very well. It took a long time to run the training because of how large the text file is. Furthermore, the model was only able to get to a loss of 1.561 with an accuracy of 54.17%.
- To be able to test how this model compared to the ABCs model, I decided to also include tests with 'a' and 'that' as well as to be able to test the model by itself. I also added 3 tests using the letter 't' but I simply did this to just measure different temperatures. Each generated sequence was 100 characters long.
- When it came the the single letter generations, when the temperature was set to 0.1, the model generated somewhat coherent structures and meaningful phrases although it repeated some words rather frequently. When the temperature was increased to 0.5, it was able to put together a much more meaningful sentence since it wasn't fully constrained and could have a bit more freedom. However, after 0.5, the generated text became incoherent and completely random with no meaning. That is why I decided to test with the single character 't' with values between 0.001 and 0.45. I found that the 0.001 text was very repetitive since it has to stick to a certain pattern very closely. Even when I raised the temperature to be 0.25 and 0.15, there was some repetition but the sentences were much more coherent and meaningful. However, when I set the temperature to be 0.45, there was no repetition and the sentence, though a bit cryptic, was much better and readable.
- While looking at predictions for the full word 'that', the output was not very good. For all four examples, the model was not able to produce one example without a spelling error. The sentences were gibberish and the more you increased the temperature, the worse it got at spelling and punctuation.  

**Difference Between the Two Sequences**
- The first major difference between the two sequences was the training losses and accuracy. The loss was much better with the training for the ABC sequence than it was for the War and Peace sequence.
- Another major difference between the two sequences was the size of the sequences. The War and Peace sequence was much larger than the ABC sequence.
- The above factors led to the difference in content generation. While generating the content for the ABCs, all it had to do was look for the next letter in the alphabet when the temperature was low, otherwise it generated a 'random' letter when the temperature was higher. In contrast, the War and Peace sequence had to learn which letter was more likely to come next out of a very large dataset. Therefore, when the temperature was lower, it would repeat phrases and words often since it was sticking to what sequence of letters came up most together. However, when you increased the temperature (not too high), it was able to also make other whole words by sampling from other letters that occur frequently next to the previous letter, although maybe not the most frequent one.

**Impact of the `temperature` Parameter**
- The `temperature` parameter affected how each model predicted their next character. Lower temperatures promote predictability, leading the model to favor highly probable next characters, which results in more stringent and repetitive outputs. Conversely, higher temperatures allow for a broader range of possibilities, leading to creative explorations at the expense of coherence, often producing strange and nonsensical variations.
- This variability illustrates the balance required when generating text: while low temperatures yield coherent but potentially uninspired outputs, high temperatures foster creativity but can drift into incoherence. Finding the right temperature is critical, especially in more complex datasets like "War and Peace," where the structure and meaning of sentences are crucial.

**Challenges and Thought Process**
- The first challenge I faced was finding the correct hyper-parameters for the models. It is crucial to the model that the hyper-parameters be good so that it can learn well and properly perform. I noticed while attempting to test the ABC sequence that the batch_size was too large at 64 and had to be reduced to 16 in order to run the whole program. However, a batch_size of 64 seemed the be the correct size needed for the War and Peace sequence. I also first ran my testing script with 20 epochs. This was an absurd thing to do considering the training time for 1 epoch using the War and Peace sequence was about 5 minutes and thus 20 epochs ended up taking 100 minutes to complete training. I eventually changed the amount of epoch down to 2 to resolve this issue.
- The second challenge involved not knowing how to use the Torch library. This was especially difficult while trying to set everything up to include the correct parameters. 

**Key Insights**
- RNNs are powerful tools for sequence data, capable of learning and predicting based on prior context. However, their performance can be significantly influenced by the training data, architecture, and hyper-parameters.
- The ability to manipulate the randomness of predictions through the temperature parameter is a powerful feature, allowing for a spectrum of creative outputs. However, a clear distinction emerged between generating coherent text and fostering creativity. The choice of temperature directly impacts whether a model should explore diverse outputs or produce safe, predictable results.
- Implementing a RNN is an iterative process that often requires multiple trials and adjustments. Understanding the dataset characteristics and how they influence model behavior is essential for successful implementation.